In [1]:
import math, os
from time import time

import torch
from spaPeakVAE import SPAPEAKVAE
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
import h5py
import scanpy as sc
from preprocess import preprocessing_atac

D:\anaconda3\envs\my\Lib\site-packages\anndata\utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
D:\anaconda3\envs\my\Lib\site-packages\anndata\utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
D:\anaconda3\envs\my\Lib\site-packages\anndata\utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
D:\anaconda3\envs\my\Lib\site-packages\anndata\utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
D:\anaconda3\envs\my\Lib\site-packages\anndata\utils.py:429: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.
  warnings.warn(msg, FutureWarning)
D:\anacon

In [3]:
# setting the hyperparameters
import argparse
parser = argparse.ArgumentParser(description='Spatial dependency-aware variational autoencoder for spatial ATAC-seq data',
                                 formatter_class=argparse.ArgumentDefaultsHelpFormatter)
parser.add_argument('--data_file', default='MISAR_seq_mouse_E15_brain_ATAC_data.h5')
parser.add_argument('--batch_size', default="auto")
parser.add_argument('--maxiter', default=5000, type=int)
parser.add_argument('--train_size', default=0.95, type=float)
parser.add_argument('--patience', default=200, type=int)
parser.add_argument('--lr', default=1e-4, type=float)
parser.add_argument('--weight_decay', default=1e-6, type=float)
parser.add_argument('--noise', default=0, type=float)
parser.add_argument('--dropoutE', default=0, type=float,
                    help='dropout probability for encoder')
parser.add_argument('--dropoutD', default=0, type=float,
                    help='dropout probability for decoder')
parser.add_argument('--encoder_layers', nargs="+", default=[1024, 128], type=int)
parser.add_argument('--GP_dim', default=5, type=int,help='dimension of the latent Gaussian process embedding')
parser.add_argument('--Normal_dim', default=8, type=int,help='dimension of the latent standard Gaussian embedding')
parser.add_argument('--decoder_layers', nargs="+", default=[128, 1024], type=int)
parser.add_argument('--dynamicVAE', default=True, type=bool, 
                    help='whether to use dynamicVAE to tune the value of beta, if setting to false, then beta is fixed to initial value')
parser.add_argument('--init_beta', default=10, type=float, help='initial coefficient of the KL loss')
parser.add_argument('--min_beta', default=4, type=float, help='minimal coefficient of the KL loss')
parser.add_argument('--max_beta', default=25, type=float, help='maximal coefficient of the KL loss')
parser.add_argument('--KL_loss', default=0.025, type=float, help='desired KL_divergence value')
parser.add_argument('--num_samples', default=1, type=int)
parser.add_argument('--fix_inducing_points', default=True, type=bool)
parser.add_argument('--grid_inducing_points', default=True, type=bool, 
                    help='whether to generate grid inducing points or use k-means centroids on locations as inducing points')
parser.add_argument('--inducing_point_steps', default=None, type=int)
parser.add_argument('--inducing_point_nums', default=None, type=int)
parser.add_argument('--fixed_gp_params', default=False, type=bool)
parser.add_argument('--loc_range', default=20., type=float)
parser.add_argument('--kernel_scale', default=20., type=float)
parser.add_argument('--model_file', default='model.pt')
parser.add_argument('--final_latent_file', default='final_latent.txt')
parser.add_argument('--denoised_counts_file', default='denoised_counts.txt')
parser.add_argument('--device', default='cuda')

args = parser.parse_args()

data_mat = h5py.File(args.data_file, 'r')
x = np.array(data_mat['X']).astype('float64') # count matrix
loc = np.array(data_mat['Pos']).astype('float64') # location information
peak_name = np.array(data_mat['Peaknames']).astype('U30') # peak names
data_mat.close()

if args.batch_size == "auto":
    if x.shape[0] <= 1024:
        args.batch_size = 128
    elif x.shape[0] <= 2048:
        args.batch_size = 256
    else:
        args.batch_size = 512
else:
    args.batch_size = int(args.batch_size)

print(args)


usage: ipykernel_launcher.py [-h] [--data_file DATA_FILE]
                             [--batch_size BATCH_SIZE] [--maxiter MAXITER]
                             [--train_size TRAIN_SIZE] [--patience PATIENCE]
                             [--lr LR] [--weight_decay WEIGHT_DECAY]
                             [--noise NOISE] [--dropoutE DROPOUTE]
                             [--dropoutD DROPOUTD]
                             [--encoder_layers ENCODER_LAYERS [ENCODER_LAYERS ...]]
                             [--GP_dim GP_DIM] [--Normal_dim NORMAL_DIM]
                             [--decoder_layers DECODER_LAYERS [DECODER_LAYERS ...]]
                             [--dynamicVAE DYNAMICVAE] [--init_beta INIT_BETA]
                             [--min_beta MIN_BETA] [--max_beta MAX_BETA]
                             [--KL_loss KL_LOSS] [--num_samples NUM_SAMPLES]
                             [--fix_inducing_points FIX_INDUCING_POINTS]
                             [--grid_inducing_points GRID_IN

SystemExit: 2

In [ ]:

scaler = MinMaxScaler()
loc = scaler.fit_transform(loc) * args.loc_range

print(x.shape)
print(loc.shape)

# We provide two ways to generate inducing point, argument "grid_inducing_points" controls whether to choice grid inducing or k-means
# One way is grid inducing points, argument "inducing_point_steps" controls number of grid steps, the resulting number of inducing point is (inducing_point_steps+1)^2
# Another way is k-means on the locations, argument "inducing_point_nums" controls number of inducing points
if args.grid_inducing_points:
    eps = 1e-5
    initial_inducing_points = np.mgrid[0:(1+eps):(1./args.inducing_point_steps), 0:(1+eps):(1./args.inducing_point_steps)].reshape(2, -1).T * args.loc_range
    print(initial_inducing_points.shape)
else:
    loc_kmeans = KMeans(n_clusters=args.inducing_point_nums, n_init=100).fit(loc)
    np.savetxt("location_centroids.txt", loc_kmeans.cluster_centers_, delimiter=",")
    np.savetxt("location_kmeans_labels.txt", loc_kmeans.labels_, delimiter=",", fmt="%i")
    initial_inducing_points = loc_kmeans.cluster_centers_

adata = sc.AnnData(x, dtype="float64")
adata.var["name"] = peak_name

adata = preprocessing_atac(adata)
peak_name = adata.var["name"].values.astype('U30')

model = SPAPEAKVAE(input_dim=adata.n_vars, GP_dim=args.GP_dim, Normal_dim=args.Normal_dim, encoder_layers=args.encoder_layers, decoder_layers=args.decoder_layers,
    noise=args.noise, encoder_dropout=args.dropoutE, decoder_dropout=args.dropoutD,
    fixed_inducing_points=args.fix_inducing_points, initial_inducing_points=initial_inducing_points, 
    fixed_gp_params=args.fixed_gp_params, kernel_scale=args.kernel_scale, N_train=adata.n_obs, KL_loss=args.KL_loss, dynamicVAE=args.dynamicVAE, 
    init_beta=args.init_beta, min_beta=args.min_beta, max_beta=args.max_beta, dtype=torch.float64, device=args.device)

print(str(model))

t0 = time()

model.train_model(pos=loc, counts=adata.X, lr=args.lr, weight_decay=args.weight_decay, 
            batch_size=args.batch_size, num_samples=args.num_samples,
            train_size=args.train_size, maxiter=args.maxiter, patience=args.patience, save_model=True, model_weights=args.model_file)
print('Training time: %d seconds.' % int(time() - t0))

final_latent = model.batching_latent_samples(X=loc, Y=adata.X, batch_size=args.batch_size)
np.savetxt(args.final_latent_file, final_latent, delimiter=",")

denoised_counts = model.batching_denoise_counts(X=loc, Y=adata.X, batch_size=args.batch_size, n_samples=25)
np.savetxt(args.denoised_counts_file, denoised_counts, delimiter=",")
